# Ludwig 2 · T0.1 — Formato, MTP y tokenizador (CPU)

Prueba, **sin gastar horas de GPU**, que un modelo con la estructura exacta de Ludwig 2 (atención completa con compuerta en todas las capas, RoPE parcial, MoE + experto compartido, embeddings atados, cabeza MTP) se puede publicar:

| Prueba | Qué verifica |
|---|---|
| A1 | Exportar y recargar en transformers da logits idénticos |
| A2 | El conversor de llama.cpp genera el GGUF, con el MTP |
| A3 | llama.cpp y transformers dan las mismas probabilidades del siguiente token |
| A4 | Las continuaciones greedy son idénticas token a token |
| A5 | La decodificación especulativa con el MTP da exactamente lo mismo, y cuántos borradores acepta |
| T1–T2 | El tokenizador se reconoce en llama.cpp y tokeniza igual que Hugging Face |

**Se corre dos veces:** con llama.cpp sin cambios (se espera que A3–A5 fallen porque llama.cpp se cae con este formato) y con el parche de Ludwig (todo debe pasar).

**Kaggle:** acelerador *None* (CPU), Internet *On*. Tarda ~15–25 min, casi todo compilando llama.cpp.

In [ ]:
REPO_URL = "https://github.com/TU_USUARIO/ludwig-lm.git"   # o "" si subiste el repo como dataset
TOKENIZER_DIR = None   # p. ej. "/kaggle/input/ludwig-tokenizer-64k" (carpeta con tokenizer.json)
LLAMA_REF = "dd6b945be2188da875ab05a259ce94c6b2736e64"   # commit con el que se obtuvieron los resultados del repo; "master" = el último

In [ ]:
import os, subprocess, sys, json
os.chdir("/kaggle/working")
if not os.path.exists("ludwig-lm"):
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "ludwig-lm"], check=True)
    else:  # el repo subido como dataset de Kaggle: se busca la carpeta que contiene ludwig_lm/
        import glob
        src = os.path.dirname(os.path.dirname(glob.glob("/kaggle/input/**/ludwig_lm/__init__.py", recursive=True)[0]))
        subprocess.run(["cp", "-r", src, "ludwig-lm"], check=True)
os.chdir("/kaggle/working/ludwig-lm")
!pip install -q "transformers==5.22.2" safetensors sentencepiece protobuf
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| CPUs", os.cpu_count(),
      "| GPUs", torch.cuda.device_count())

## 1. Descargar y compilar llama.cpp (sin cambios)

In [ ]:
LLAMA = "/kaggle/working/llama.cpp"
if not os.path.exists(LLAMA):
    !git init -q {LLAMA} && git -C {LLAMA} remote add origin https://github.com/ggml-org/llama.cpp
    !git -C {LLAMA} fetch -q --depth 1 origin {LLAMA_REF} && git -C {LLAMA} checkout -q FETCH_HEAD
!git -C {LLAMA} log -1 --format="llama.cpp %H (%cd)"
!cmake -S {LLAMA} -B {LLAMA}/build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF > /dev/null
!cmake --build {LLAMA}/build -j {os.cpu_count()} --target llama-server llama-tokenize 2>&1 | tail -2

## 2. Prueba con llama.cpp sin cambios (se espera que A3–A5 fallen)

In [ ]:
tok = f"--tokenizer {TOKENIZER_DIR}" if TOKENIZER_DIR else ""
!python scripts/smoke_format.py --llama-cpp {LLAMA} --out results/t0_1_stock.json {tok} 2>&1 | grep -v -i warn | tail -25

## 3. Prueba con el parche de Ludwig (todo debe pasar)

In [ ]:
!python scripts/smoke_format.py --llama-cpp {LLAMA} --patch --out results/t0_1_patched.json {tok} 2>&1 | grep -v -i warn | tail -25

## 4. Resumen

In [ ]:
import pandas as pd
rows = []
for name in ["stock", "patched"]:
    r = json.load(open(f"results/t0_1_{name}.json"))
    for k, v in r.items():
        if isinstance(v, dict) and "pass" in v:
            rows.append({"llama.cpp": name, "prueba": k, "pasa": "✅" if v["pass"] else "❌",
                         "detalle": {kk: vv for kk, vv in v.items() if kk not in ("pass", "log_tail", "error", "mismatches")}})
    if r.get("patch_note"):
        print(f"{name}: {r['patch_note']}")
pd.set_option("display.max_colwidth", 120)
display(pd.DataFrame(rows))

## Qué hacer con el resultado

- Descarga `results/t0_1_stock.json` y `results/t0_1_patched.json` (panel *Output*) y súbelos al repo: son la evidencia del diseño.
- Si corriste con `TOKENIZER_DIR`, mira `T1_ludwig64k_pretokenizer` y `T2_ludwig64k_parity`. Si T1 falla, el regex del tokenizador no es uno que llama.cpp implemente y hay que cambiarlo **antes** del tronco.
- Si con `LLAMA_REF = "master"` la prueba sin parche ya pasa, llama.cpp arregló el fallo por su cuenta y el parche sobra.
- El parche y el texto del PR para llama.cpp están en `patches/`.